<a href="https://colab.research.google.com/github/mariefarhat/Lebanon-Conflict-GIS/blob/main/Lebanon_Conflict.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lebanese Conflict Hot-Spots and Shelter Access During the 2024 War
Marie Farhat

May 2026, The University of Chicago

## Background

In the Fall of 2024, violence between Hezbollah and Israel escalated following a year of hostilities in the wake of the Hamas-led October 7 attack. Hezbollah ("Party of God" in Arabic) formed in 1982 in response to the Israeli invasion of Lebanon. Hezbollah is a Shia terrorist group and political party with prominent presence in Southern Lebanon, the Beqaa Valley in eastern Lebanon, and in Dahieh (southern suburbs of Beirut, near the airport). During this period, Hezbollah's leader Hassan Nasrallah, was assassinated by the Israeli military (September 27, 2024) in Dahieh. Israel's goal in escalating the war was (and continues to be) to weaken Hezbollah, which has perpetuated violence at Israel's northern border over decades, making the region unsafe for Israeli civilians.


Lebanese and Israeli civilians continue to be caught in the crosshairs of this generational conflict-- regardless of their support of the war. **This project aims to examine the geography of the conflict and government response on the Lebanese side through geospatial analysis.** My motivation for this study is personal. I am Lebanese-American, born to parents who grew up during the Lebanese Civil War. I experienced the 2006 July War between Hezbollah and Israel (just like [Anthony Bourdain](https://www.rottentomatoes.com/tv/anthony_bourdain_no_reservations/s02/e14)!) and was present during the hostilities in summer 2024, before things took a turn for the worse. Many of my relatives still live there under constant threat of violence. Having never taken a course on Lebanon nor used Lebanese data in an academic setting, I wanted to **combine my lived experience with the data science techniques I learned through UChicago CAPP and Data & the State to answer my research questions:**


*   **What is the geography of conflict in Lebanon during the 2024 war?**
*   **How accessible are government-designated shelters in Dahieh?**



## Imports

In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import requests
import folium
from libpysal.weights import Queen
from esda.moran import Moran_Local
from shapely.geometry import box
from pathlib import Path
import zipfile
from pprint import pprint

## Lebanon's Administrative Regions
As a first step for geospatial analysis, let's load some maps of Lebanon from the [Humanitarian Data Exchange](https://data.humdata.org/dataset/cod-ab-lbn) (HDX).

We'll need to project these maps in two coordinate reference systems: one for analysis to measure distance in metric units (EPSG:32636) and one for map visualization (EPSG:4326) in degrees.

In [ ]:
analysis_crs = "EPSG:32636"
map_crs = "EPSG:4326"

### Loading the Data
Let's query the HDX website for Lebanese administrative boundary files (specifically geoJSON extensions) using the API
https://hapi.humdata.org/docs documentation.

In [ ]:
# Query HDX API for Lebanon Subnational Administrative Boundaries
api_url = f"https://data.humdata.org/api/3/action/package_show?id=cod-ab-lbn"

response = requests.get(api_url)
response.raise_for_status()

data = response.json()

resources = data["result"]["resources"]

res_df = pd.DataFrame([
    {
        "name": r["name"],
        "format": r.get("format"),
        "url": r["url"]
    }
    for r in resources
])

# See the available files
print("Available files:")
print(res_df[["name", "format"]])

# Find the geojson zip file and download
geojson_row = res_df[res_df["format"].str.contains("geojson", case=False, na=False)].iloc[0]

geojson_url = geojson_row["url"]

print("\nDownloading:")
print(geojson_url)

zip_path = Path("lebanon_admin_geojson.zip")

download = requests.get(geojson_url)
download.raise_for_status()

zip_path.write_bytes(download.content)

print("\nDownload complete.")

# Extract files to a directory
extract_dir = Path("lebanon_geojson")

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)

print("\nExtraction complete.")

# See list of extracted geojson files
pprint(list(extract_dir.glob("*")))

### Data Cleaning
What do these administrative boundaries correspond to?


*   admin0 = Country boundary
*   admin1 = Governorates/Mohafaza
*   admin2 = Districts/Cazas/Kadaa
*   admin3 = Municipalities/Subdistricts/Cadastral
*   capitals = Administrative capitals
*   adminpoints = Admin region centroids
*   adminlines = Boundary lines

For the purpose of this analysis, we will ignore the files with '_em' which are for emergency/humanitarian operational boundaries commonly used by the UN.
Now, let's read the GeoJSON files into Pandas:

In [ ]:
admin_files_dir = Path("lebanon_geojson")

# Files to keep
admin_files = {
    "admin0": "lbn_admin0.geojson",
    "admin1": "lbn_admin1.geojson",
    "admin2": "lbn_admin2.geojson",
    "admin3": "lbn_admin3.geojson",
    "capitals": "lbn_admincapitals.geojson",
    "adminpoints": "lbn_adminpoints.geojson",
    "adminlines": "lbn_adminlines.geojson",
}

# make non-geometry columns JSON/Folium-safe
def make_json_safe_gdf(gdf):
    gdf = gdf.copy()

    for col in gdf.columns:
        if col != "geometry":
            gdf[col] = gdf[col].astype(str)

    return gdf

# Read, reproject, and clean into dictionary of GeoDataFrames
admin_gdfs = {
    name: make_json_safe_gdf(
        gpd.read_file(admin_files_dir / filename).to_crs(map_crs)
    )
    for name, filename in admin_files.items()
}

# Access each gdf
admin0_gdf = admin_gdfs["admin0"]
admin1_gdf = admin_gdfs["admin1"]
admin2_gdf = admin_gdfs["admin2"]
admin3_gdf = admin_gdfs["admin3"]
capitals_gdf = admin_gdfs["capitals"]
adminpoints_gdf = admin_gdfs["adminpoints"]
adminlines_gdf = admin_gdfs["adminlines"]

# Test
admin1_gdf.head(2)

### Mapping the Regions
Let's examine the Admin 1, 2 and 3 maps.

In [ ]:
# Center the map on Lebanon using Admin 0
minx, miny, maxx, maxy = admin0_gdf.total_bounds
center = [(miny + maxy) / 2, (minx + maxx) / 2]

admin_map = folium.Map(
    location=center,
    zoom_start=8,
    tiles="CartoDB positron"
)

def add_admin_layer(m, gdf, layer_name, hover_cols, color, show=False):
    folium.GeoJson(
        gdf,
        name=layer_name,
        tooltip=folium.GeoJsonTooltip(
            fields=hover_cols,
            aliases=[col.replace("_", " ").title() for col in hover_cols],
            sticky=True
        ),
        style_function=lambda feature, color=color: {
            "fillColor": color,
            "color": "black",
            "weight": 1,
            "fillOpacity": 0.25,
        },
        highlight_function=lambda feature: {
            "weight": 3,
            "fillOpacity": 0.5,
        },
        show=show
    ).add_to(m)

add_admin_layer(admin_map, admin1_gdf, "Admin 1", ["adm1_name"], "green", show=True)
add_admin_layer(admin_map, admin2_gdf, "Admin 2", ["adm1_name", "adm2_name"], "blue", show=False)
add_admin_layer(admin_map, admin3_gdf, "Admin 3", ["adm1_name", "adm2_name", "adm3_name"], "red", show=False)

folium.LayerControl(collapsed=False).add_to(admin_map)

admin_map

# Part 1: Visualizing Conflict
This section of the notebook examines the geographic nature of conflict during the 2024 war and identifies conflict hot-spots using Local Moran's I.
I hypothesize that there will be a stark difference across regions, and that hot-spots will be heavily concentrated in the southern and eastern parts of the country, along with the southern suburbs of Beirut. This hypothesis was formulated based on my own experience, anecdotal evidence, and news coverage of the war.

## Conflict Data

The conflict data used in this analysis comes from [ACLED](https://acleddata.com/conflict-data/data-export-tool?viewsreference%5Breload%5D=eJxdi8EKwjAQRP9lzx7UIk3zKyLLSta6kK4hWVqK9N9NyaHgYQ4z780XWOkZOWBhM9GxgL8_TpAosxrWiK1oa2Lw-0hjpvSGf0FCxdfOuduBXsIxoNK0X1uZhZdDyDxLkY-292UYOtdX2lQxnjBwNAJ_3n69bjrE&event_date_from=2023-10-01&event_date_to=2025-05-22&country%5B422%5D=422&fields_on%5Bpopulation_best%5D=population_best&fields_on%5Bevent_id_cnty%5D=event_id_cnty) (Armed Conflict Location & Event Data), an independent nonprofit organization that collects and publishes data on political violence, conflict, demonstrations, and strategic developments worldwide. ACLED records discrete incidents of political violence or demonstrations that occur at a specific location and date. Each event includes standardized information such as:

* event date
* geographic coordinates
* administrative region
* event type and sub-event type
* actors involved
* fatalities
* descriptive notes
* source citations

You can find more information on ACLED data definitions [here](https://acleddata.com/sites/default/files/wp-content-archive/uploads/2021/11/ACLED_Event-Definitions_v1_April-2019.pdf) and [here](https://acleddata.com/methodology/cast-methodology).

### Loading the Data
Accessing ACLED data requires a user to have account. For this reason, I have downloaded a CSV of events in Lebanon from 2023 to 2025 and saved within a GitHub repository to mitigate access and API issues.

In [ ]:
conflict_df = pd.read_csv("https://raw.githubusercontent.com/mariefarhat/Lebanon-Conflict-GIS/refs/heads/main/ACLED%20Data_2026-05-22.csv")
conflict_gdf = gpd.GeoDataFrame(
    conflict_df,
    geometry=gpd.points_from_xy(conflict_df.longitude, conflict_df.latitude),
    crs="EPSG:4326"
)

conflict_gdf.head(2)

### Data Cleaning
What kinds of events are tracked in the ACLED dataset?
In this case, we care about diving into events that would displace people and drive them towards shelters, such as **Battles, Explosions/Remote violence,** and **Violence against civilians**.

In [ ]:
event_counts = (
    conflict_gdf
    .groupby(["event_type", "sub_event_type"])
    .size()
    .reset_index(name="count")
    .sort_values(
        by=["event_type", "count"],
        ascending=[True, False]
    )
)

event_counts.sort_values(by=["event_type", "count"], ascending=[True, False])

While we're here, let's also filter the dataset by dates corresponding to the official 2024 Israel-Lebanon War.

In [ ]:
# Filter for desired ACLED events
event_types = ["Battles", "Explosions/Remote violence", "Violence against civilians"]
conflict_events_gdf = conflict_gdf[conflict_gdf["event_type"].isin(event_types)]

# Filter event_date between 9/23/24 (official) and 11/27/24 (ceasefire)
conflict_events_gdf = conflict_events_gdf[
    (conflict_events_gdf["event_date"] >= "2024-09-23") &
    (conflict_events_gdf["event_date"] <= "2024-11-27")
]

# Let's see the breakdown of the data
conflict_events_gdf["sub_event_type"].value_counts()

### Visualizing Conflict with a Choropleth Map
Let's observe the concentration of conflict by Admin 3 region.

In [ ]:
# Consistent CRS
conflict_events_gdf = conflict_events_gdf.to_crs(admin3_gdf.crs)

# Spatially join conflict events within admin3 polygons
conflict_admin3 = gpd.sjoin(
    conflict_events_gdf,
    admin3_gdf,
    how="left",
    predicate="within"
)

# Get the number of conflict events per admin3 region
conflict_counts = (
    conflict_admin3
    .groupby("index_right")
    .size()
    .reset_index(name="conflict_count")
)

# Merge shelter counts back to the admin3 regions
admin3_conflict_choro = admin3_gdf.merge(
    conflict_counts,
    left_index=True,
    right_on="index_right",
    how="left"
)

admin3_conflict_choro["conflict_count"] = admin3_conflict_choro["conflict_count"].fillna(0)

It looks like Baalbek, Hermel, Beirut's southern suburbs, and Southern Lebanon (namely el Nabatieh and Sour) have the highest number of conflict incidents in the map below.

In [ ]:
conflict_choro = admin3_conflict_choro.to_crs(map_crs).copy()

# Fill missing values
conflict_choro["conflict_count"] = (
    conflict_choro["conflict_count"]
    .fillna(0)
)

# Create unique id
conflict_choro = conflict_choro.reset_index(drop=True)
conflict_choro["map_id"] = conflict_choro.index

conflict_map = folium.Map(
    location=center,
    zoom_start=8,
    tiles="CartoDB positron"
)

# Choropleth
folium.Choropleth(
    geo_data=conflict_choro,
    data=conflict_choro,
    columns=["map_id", "conflict_count"],
    key_on="feature.properties.map_id",
    fill_color="BuPu",
    fill_opacity=0.9,
    legend_name="Conflict Count",
    name="Conflict Count"
).add_to(conflict_map)

# Tooltip
folium.GeoJson(
    conflict_choro,
    style_function=lambda feature: {
        "fillOpacity": 0,
        "color": "black",
        "weight": 0.2
    },
    tooltip=folium.GeoJsonTooltip(
        fields=["adm3_name", "adm2_name", "adm1_name", "conflict_count"],
        aliases=["Admin 3", "Admin 2", "Admin 1", "Conflict Count"],
        sticky=True
    )
).add_to(conflict_map)

conflict_map

## Conflict Hot-Spots with Local Moran's I
To better understand how the 2024 ACLED conflict data is distributed across Lebanon, I conducted a **Local Moran’s I hotspot analysis** using conflict event counts aggregated at the Admin 3 level. This analysis looks at whether neighboring areas are experiencing similar levels of conflict, helping reveal broader geographic patterns across the country.

The analysis compares each Admin 3 region to the areas directly surrounding it, defining neighbors as regions that share either a border or corner. Based on these relationships, each area was classified into one of four categories:
* **High-High** areas represent concentrated conflict hotspots surrounded by other
high-conflict areas
* **Low-Low** areas represent relatively stable regions with consistently lower conflict activity
* **High-Low **or** Low-High** areas represent geographic outliers whose conflict levels differ from neighbors.

Statistical significance testing was also used to help distinguish meaningful spatial patterns from clusters that may have occurred by chance.

The code below was inspired and helped by the documentation found in [here](https://pysal.org/notebooks/viz/splot/esda_morans_viz.html).



In [ ]:
# Build spatial weights between neighboring admin3 polygons
w_conflict = Queen.from_dataframe(admin3_conflict_choro)
w_conflict.transform = "r"

# run Local Moran's I and store quadrant classificiation and statistical significance
moran_conflict = Moran_Local(
    admin3_conflict_choro["conflict_count"],
    w_conflict
)

admin3_conflict_choro["quadrant"] = moran_conflict.q
admin3_conflict_choro["significant"] = moran_conflict.p_sim < 0.05

# create labels for each quadrant within a cluster classification column
quadrants = [
    admin3_conflict_choro["quadrant"] == 1,
    admin3_conflict_choro["quadrant"] == 2,
    admin3_conflict_choro["quadrant"] == 3,
    admin3_conflict_choro["quadrant"] == 4
]
labels = ["High-High", "Low-High", "Low-Low", "High-Low"]
admin3_conflict_choro["cluster"] = np.select(quadrants, labels, default="Not Significant")

In [ ]:
cluster_colors = {
    "High-High": "#d7191c",
    "High-Low": "#fdae61",
    "Low-High": "#abd9e9",
    "Low-Low": "#2c7bb6",
    "Not Significant": "lightgray"
}

category_order = list(cluster_colors.keys())

cluster_gdf = admin3_conflict_choro.copy()

cluster_gdf["cluster"] = pd.Categorical(
    cluster_gdf["cluster"],
    categories=category_order,
    ordered=True
)

fig, ax = plt.subplots(figsize=(10, 10))

cluster_gdf.plot(
    ax=ax,
    color=cluster_gdf["cluster"].map(cluster_colors),
    edgecolor="black",
    linewidth=0.2
)

legend_handles = [
    mpatches.Patch(color=color, label=label)
    for label, color in cluster_colors.items()
]

ax.legend(
    handles=legend_handles,
    title="Moran's I Clusters",
    loc="upper left",
    frameon=True
)

ax.set_title("Local Moran's I Conflict Clusters", fontsize=14)
ax.set_axis_off()

plt.show()

## Reflection

The above hot-spot map confirms my experience in 2024 before the war escalated. It is clear from the cluster map that avoiding the south and eastern parts of Lebanon in 2024 meant that you could be relatively safe from most airstrikes. However, southern Beirut and Dahieh are unavoidable if you're flying into the country (the airport is located there). The tight clustering pattern also supports why my relatives in northern Beirut and its southeastern suburbs feel "relatively" safe (they still live and work within a few kilometers of the action, but it's a completely different perspective and reality there).

Another way to think about this map: the hot-spots are the Admin 3 areas that Israel believes hold Hezbollah targets. These also happen to be predominantly Shia areas. While confirming this empirically is beyond the scope of this notebook (I don't know where terror cells are located, and Lebanon hasn't conducted an official census in nearly a century), you can see for yourself how it correlates with this religious map of Lebanon by municipality according to elections data ([Wikipedia](https://en.wikipedia.org/wiki/Religion_in_Lebanon)).

![Lebanon Religion Map](https://raw.githubusercontent.com/mariefarhat/Lebanon-Conflict-GIS/main/Lebanon_religion_map_by_municipality.png)

# Part 2: Shelter Access
The Lebanese Government responded to the war by activating shelters across the country to house internally displaced persons. This section of the notebook will examine shelter access in Dahieh as it relates to evacuation procedures. I hypothesize that Dahieh might be underserved and not have sufficient government designated shelters. This will be tested by identifying the shelters in Dahieh and finding how many buildings are within a walking distance to a shelter.

## Government Designated Shelters

This report on [Government Designated Shelters](https://api.beiruturbanlab.com/Content/uploads/Articles/796~Report-Gov-Designated-Shelters.pdf) documents collective shelters activated across Lebanon during the 2024 displacement crisis, primarily public schools and technical institutes repurposed to house internally displaced persons during the Israeli escalation and war. Created by the [Beirut Urban Lab](https://beiruturbanlab.com/en/Details/2005/sheltering-centers-in-public-schools-and-technical-institutes-in-lebanon), the related [interactive map](https://experience.arcgis.com/experience/af252d852fd144ad98242eba8b6d60b3/page/English) includes shelter locations, capacities, operational status, and occupancy trends over time.

The report highlights that shelters functioned as emergency “overflow infrastructure,” rapidly expanding as displacement surged and concentrating heavily in major receiving areas such as Beirut and Mount Lebanon. For this project, the dataset provides a spatial foundation for analyzing shelter accessibility and emergency accommodation capacity in Dahieh, an area heavily affected by evacuation orders, bombardment, and displacement during the conflict.

### Loading the data
Because the shelter dataset was not directly downloadable, I accessed the ArcGIS FeatureServer endpoint underlying the interactive Esri web map by inspecting the map’s network requests in the browser Developer Tools. I then queried the FeatureServer directly using its REST API, requesting all shelter records and geometries in GeoJSON format. The response was loaded into a GeoPandas GeoDataFrame for subsequent spatial analysis and mapping, with latitude and longitude coordinates extracted from the shelter point geometries.

In [ ]:
url = "https://services3.arcgis.com/tuNLpt6Wfhd22qmO/arcgis/rest/services/SchoolForEmergencyPlan_Map/FeatureServer/0/query?where=1%3D1&outFields=*&returnGeometry=true&outSR=4326&f=geojson"

shelter_gdf = gpd.read_file(url)

shelter_gdf["lon"] = shelter_gdf.geometry.x
shelter_gdf["lat"] = shelter_gdf.geometry.y

print(shelter_gdf.head())

### Data cleaning

As we can see in the shelter data, some columns and values are in Arabic. Most of us don't have an Arabic keyboard handy, so let's perform some translations.

In [ ]:
# Let's inspect the columns
shelter_gdf.columns

In [ ]:
# Translate columns from Arabic
shelter_gdf = shelter_gdf.rename(columns={
    "وضع_المدرسة": "school_status",
    "مركز_ايواء": "shelter_center",
    "رقم_الهاتف": "phone_number",
    "محافظة": "governorate_arabic"
})

shelter_gdf.head(3)

In [ ]:
# Now that we've translated the columns, let's see what's inside
shelter_gdf.info()

In [ ]:
# What's in school_status?
shelter_gdf["school_status"].value_counts(dropna=False)

The school_status column has two values in Arabic: مقفلة / 'maqfula' meaning the school is closed (literal translation is 'locked'), and موجودة مرتين / 'mowjouda martein' meaning the record is a duplicate.

To clean our data, let's take out the duplicate records, (but keep the closed schools in our list as they can be repurposed in the 'fix as you go' model outlined in the report).

In [ ]:
shelter_gdf = (
    shelter_gdf
    .sort_values(
        by="school_status",
        key=lambda s: s.eq("موجودة مرتين")
    )
    .drop_duplicates(subset="School_Name", keep="first")
)

Finally, as a backup, let's save our cleaned shelter data into a GeoJSON file

In [ ]:
shelter_gdf.to_csv("lebanon_shelters.csv", index=False)
shelter_gdf.to_file("lebanon_shelters.geojson", driver="GeoJSON")

### Mapping the Shelters
Let's see where the shelters are located and recreate the ArcGIS interactive map from Beirut Urban Lab.

In [ ]:
shelter_gdf = shelter_gdf.to_crs(map_crs)

shelter_map = folium.Map(
    location=center,
    zoom_start=8,
    tiles="CartoDB positron"
)

# Add admin layers
add_admin_layer(shelter_map, admin1_gdf, "Admin 1", ["adm1_name"], "green", show=True)
add_admin_layer(shelter_map, admin2_gdf, "Admin 2", ["adm1_name", "adm2_name"], "blue", show=False)
add_admin_layer(shelter_map, admin3_gdf, "Admin 3", ["adm1_name", "adm2_name", "adm3_name"], "red", show=False)

# Add shelter points
shelter_layer = folium.FeatureGroup(name="Shelters", show=True)

for _, row in shelter_gdf.iterrows():

    if row.geometry is None:
        continue

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=3,
        color="black",
        fill=True,
        fill_color="white",
        fill_opacity=1,
        weight=1
    ).add_to(shelter_layer)

shelter_layer.add_to(shelter_map)

folium.LayerControl(collapsed=False).add_to(shelter_map)

shelter_map

## Evacuation Warnings
### (This part contains context rather than code)
The Israeli military warns Lebanese residents of incoming strikes and evacuation notices through texts, [air-dropped leaflets](https://www.nytimes.com/2026/03/13/world/middleeast/israeli-leaflets-over-beirut-invoke-gazas-destruction-and-stoke-fear.html), social media, and televised statements. While not every strike is forewarned, many warnings are issued at short notice. In one documented instance during the 2024 war, a warning was issued in the middle of the night less than 30 minutes before strikes began, according to [Amnesty International](https://www.amnesty.org/en/latest/news/2024/10/lebanon-israels-evacuation-warnings-for-civilians-misleading-and-inadequate/).

#### Example Warning (with images)

This warning was shared by the Israeli military via X on September 27th, 2024. The 500m buffer radius in the image actually only covers a 135m radius. Below is a translation of the warning message:
> Warning to the residents of Dahieh:
To the residents of the Laylaki neighborhood and those living in the building Mitar Shdid and the surrounding buildings shown on the map.
You are located near facilities belonging to Hezbollah.
For your safety and the safety of your families, you are required to evacuate these buildings immediately and move at least 500 meters away, as shown on the map.

![Evacuation Warning](https://raw.githubusercontent.com/mariefarhat/Lebanon-Conflict-GIS/main/evac_warning.jpeg)

A correct 500m buffer zone would cover much more space, as seen in this map created by Ahmad Baydoun (Amnesty International):

![Corrected Evacuation Warning](https://raw.githubusercontent.com/mariefarhat/Lebanon-Conflict-GIS/main/evac_warning_corrected.jpg)

## Beirut & Suburban Buildings

To measure shelter accessibility in Dahieh, we'll use the buildings geometries provided by Beirut Urban Lab through their [Beirut Built Environment Database](https://beirut-urban-lab-open-data-platform-aub.hub.arcgis.com/). These buildings will be used to calculate how many are within a walking distance to the a government designated shelter.

### Loading the Data

In [ ]:
# Greater Beirut buildings API
greater_beirut_api = (
    "https://services3.arcgis.com/tuNLpt6Wfhd22qmO/arcgis/rest/services/GreaterBeirut_Basemap/FeatureServer/0/query?outFields=*&where=1%3D1&f=geojson"
)

# Helper function with pagination
def load_buildings(url, batch_size=2000):
    base_url = url.split("?")[0]

    all_features = []
    offset = 0

    while True:
        params = {
            "where": "1=1",
            "outFields": "*",
            "f": "geojson",
            "resultOffset": offset,
            "resultRecordCount": batch_size
        }

        response = requests.get(base_url, params=params)
        response.raise_for_status()

        data = response.json()
        features = data.get("features", [])

        if len(features) == 0:
            break

        all_features.extend(features)

        print(f"Downloaded {len(all_features)} features")

        offset += batch_size

    gdf = gpd.GeoDataFrame.from_features(
        all_features,
        crs=map_crs
    )

    return gdf


# Create output directory
output_dir = Path("buildings_geojson")
output_dir.mkdir(exist_ok=True)

# Load and save Greater Beirut buildings
greater_beirut_buildings = load_buildings(greater_beirut_api)

output_path = output_dir / "greater_beirut_buildings.geojson"

greater_beirut_buildings.to_file(
    output_path,
    driver="GeoJSON"
)

print(f"Saved: {output_path}")

greater_beirut_gdf = gpd.read_file("buildings_geojson/greater_beirut_buildings.geojson")

### Data Cleaning

In [ ]:
# Let's inspect the Greater Beirut columns and CRS
print(greater_beirut_gdf.columns)
print(greater_beirut_gdf.crs)
greater_beirut_gdf.head(3)

Mohafaza is our admin 1 region, Kadaa is our admin 2, and Cadastral is our admin 3. Now let's look at what's in some of these columns, like DataSource, Status_2022, and Development Status. We'll focus on the **Baabda** 'Kadaa' as it represents the southern suburbs of Beirut, where Dahieh is located.

In [ ]:
baabda_gdf = greater_beirut_gdf[greater_beirut_gdf["Kadaa"] == "Baabda"]
pprint(baabda_gdf["DataSource"].value_counts())
print("\n")
pprint(baabda_gdf["Status_2022"].value_counts())
print("\n")
pprint(baabda_gdf["Type"].value_counts())
print("\n")
pprint(baabda_gdf["Cadastral"].value_counts())

It seems like we have quite a bit of 'Not Available' values for our buildings, and many haven't been categorized by type or residential status. This makes data cleaning hard, so for the purpose of this notebook, all recorded buildings within a region will be used for shelter-access analysis.

Kfar Chima (Cadastral, where I am from) is not one of the towns considered part of **Dahieh**, so let's whittle Baabda down to the towns / admin3 areas we want to analyze.

In [ ]:
dahieh = [
    "Chiyah",
    "Bourj El Brajneh",
    "Hadath Beyrouth",
    "Haret Hreik",
    "Tahouitat El Ghadir",
    "Laylaké",
    "Furn Ech Chebbak"
]
dahieh_gdf = baabda_gdf[baabda_gdf["Cadastral"].isin(dahieh)]

## Shelter Access in Dahieh

### Data Preparation


First, we need to gather our buildings and shelters GeoDataFrames and reproject to use distance in meters rather than degrees, using **analysis_crs** that we set at the start of the notebook when we imported the Lebanon administrative regions.


In [ ]:
buildings_dahieh = dahieh_gdf.to_crs(analysis_crs)
shelters_dahieh = shelter_gdf.copy().to_crs(analysis_crs)
print(analysis_crs)

Now let's set a bounding box around Dahieh and clip the shelters inside the area.

In [ ]:
# Bounding box
minx, miny, maxx, maxy = buildings_dahieh.total_bounds
dahieh_boundary = box(
    minx,
    miny,
    maxx,
    maxy
)

# Clip shelters using reprojected gdfs
shelters_dahieh = gpd.clip(shelters_dahieh, dahieh_boundary)

### How do we define shelter access?

Let's imagine a situation where you're living in Dahieh and trying to evacuate your home within a short window (~ 30 minutes). You can't go very far in your car because the roads are blocked by rubble or traffic. Perhaps you have children, or elderly parents with you, and it's dangerous to camp outside (though many resort to this). For the moment, having a roof over your head is a priority before you can find long-term sheltering in a safer region.

We can define shelter access using straight-line distance buffers measured in meters, as with our analysis CRS. Using the [How Long To Walk](https://howlongtowalk.org/how-far-can-i-walk/in-30-minutes) calculator, and estimating an 'leisurely' pace, we can establish a pace of 1.6km (= 1600m) in 30 minutes. I use slower pace because I expect some people may be injured, traversing difficult pathways covered in rubble, fighting crowds, carrying heavy belongings, or guiding children and the elderly.

In [ ]:
# Straight line distance buffer (1600 meters)
buffer_distance = 1600

buffer_geom = shelters_dahieh.buffer(buffer_distance).union_all()

shelter_buffers_gdf = gpd.GeoDataFrame(
    [{
        "distance_m": buffer_distance,
        "geometry": buffer_geom
    }],
    crs=analysis_crs
)

In [ ]:
# Let's see how many buildings in Dahieh are within 1.6km of a shelter
results = []

for _, row in shelter_buffers_gdf.iterrows():
    building_count = buildings_dahieh.intersects(row.geometry).sum()

    results.append({
        "distance_m": row["distance_m"],
        "building_count": building_count,
        "percent_of_buildings": (
            building_count / len(buildings_dahieh) * 100
        )
    })

buffer_summary = pd.DataFrame(results)
buffer_summary

It appears that Dahieh has really high coverage of shelters within a relatively short distance, with nearly all buildings within 1.6km of a shelter.
Let's see what this looks like as a map.

### Mapping Shelter Access

In [ ]:
# Reproject to our map_crs using lat lon
buffers_folium = shelter_buffers_gdf.to_crs(map_crs)
shelters_folium = shelters_dahieh.to_crs(map_crs)

# Create building centroids (faster than mapping each polygon) and reproject
building_pts = buildings_dahieh.copy()
building_pts["geometry"] = building_pts.geometry.centroid
building_pts = building_pts.to_crs(map_crs)

# Center map on shelters
center = [
    shelters_folium.geometry.y.mean(),
    shelters_folium.geometry.x.mean()
]

dahieh_shelter_map = folium.Map(
    location=center,
    zoom_start=14,
    tiles="CartoDB positron"
)

# Building centroids
for _, row in building_pts.iterrows():

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=1,
        color="orange",
        fill=True,
        fill_color="orange",
        fill_opacity=0.5,
        opacity=0.5,
    ).add_to(dahieh_shelter_map)

# Shelters
for _, row in shelters_folium.iterrows():

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=4,
        color="blue",
        fill=True,
        fill_color="blue",
        fill_opacity=1,
        opacity=1
    ).add_to(dahieh_shelter_map)

# 1.6km shelter buffer
folium.GeoJson(
    buffers_folium,
    name="1.6km shelter buffer",
    style_function=lambda feature: {
        "fillColor": "lightblue",
        "color": "blue",
        "weight": 1,
        "fillOpacity": 0.25,
    },
    highlight_function=None
).add_to(dahieh_shelter_map)

folium.LayerControl().add_to(dahieh_shelter_map)

dahieh_shelter_map

## Reflection
Based on this map and the data assumptions, it appears the Dahieh had great distance-based access to shelters. **Important caveat:** shelter capacity and adequacy (which we don't have) are deeper measures of access, showing how many families or individuals can be hosted at a given time. Furthermore, Dahieh residents may not want to seek shelter in an area prone to attacks, and thus flee to neighboring regions that appear on the 'low' end of the Moran's I quadrant. The straight-line buffers don't account for road blockages.

# References


> Humanitarian Data Exchange (HDX). Lebanon – Subnational administrative boundaries. United Nations Office for the Coordination of Humanitarian Affairs. https://data.humdata.org/dataset/cod-ab-lbn

> Armed Conflict Location & Event Data Project (ACLED). ACLED conflict data for Lebanon [Data set]. https://acleddata.com/conflict-data/data-export-tool

> PySAL Developers. Moran’s I visualization for exploratory spatial data analysis. PySAL. https://pysal.org/notebooks/viz/splot/esda_morans_viz.html

> Al-Harithy, H., & Haddad, W. (2026, April). Government-designated shelters in Lebanon. Beirut Urban Lab, American University of Beirut. https://api.beiruturbanlab.com/Content/uploads/Articles/796~Report-Gov-Designated-Shelters.pdf

> Beirut Urban Lab. (2024, October 1). Sheltering centers in public schools and technical institutes in Lebanon. American University of Beirut. https://beiruturbanlab.com/en/Details/2005/sheltering-centers-in-public-schools-and-technical-institutes-in-lebanon

> Beirut Urban Lab Open Data Platform. American University of Beirut. https://beirut-urban-lab-open-data-platform-aub.hub.arcgis.com/

> Amnesty International. (2024, October). Lebanon: Israel’s evacuation warnings for civilians misleading and inadequate. https://www.amnesty.org/en/latest/news/2024/10/lebanon-israels-evacuation-warnings-for-civilians-misleading-and-inadequate/

> How far can I walk in 30 minutes? https://howlongtowalk.org/how-far-can-i-walk/in-30-minutes

# AI Disclosure
I used Gemini in Google Colab to debug my code using the 'Explain this error' feature. I also used Gemini to help structure my API requests when I ran into pagination issues, and resolve some ineffective code (for example, I kept crashing my instance when trying to visualize the thousands of building polygons, and the LLM recommended using centroids for more stable results).